# Keyboard Coordinator

> Global coordinator for hierarchical keyboard system management with single-listener event dispatch.

In [ ]:
#| default_exp js.coordinator

In [ ]:
#| export
from __future__ import annotations

## Coordinator Setup

The coordinator is a global singleton (`window.kbCoordinator`) that manages registration, hierarchy, and event dispatch for all keyboard systems on a page. Each keyboard system registers its handler instead of adding its own `document.keydown` listener. The coordinator owns the single listener and routes events through the parent-child hierarchy: deepest active child handles first, unhandled events bubble up.

In [ ]:
#| export
def js_coordinator_setup() -> str: # JavaScript coordinator singleton code
    """Generate the global keyboard coordinator singleton."""
    return '''
// === Keyboard Coordinator ===
if (!window.kbCoordinator) {
    window.kbCoordinator = (function() {
        const systems = {};       // systemId -> { handle, initialize, getState, config, onActivate, onDeactivate }
        const children = {};      // parentId -> [childId, ...]
        const parents = {};       // childId -> parentId
        const activeChild = {};   // parentId -> childId or null
        let rootSystems = [];     // systems with no parent
        let _listener = null;     // single keydown listener reference

        // --- Registration ---

        function register(systemId, handler) {
            systems[systemId] = handler;
            // Default to root until setParent is called
            if (!parents[systemId]) {
                if (!rootSystems.includes(systemId)) {
                    rootSystems.push(systemId);
                }
            }
            _ensureListener();
        }

        function unregister(systemId) {
            delete systems[systemId];
            // Remove from root list
            rootSystems = rootSystems.filter(id => id !== systemId);
            // Remove from parent's children
            const parentId = parents[systemId];
            if (parentId && children[parentId]) {
                children[parentId] = children[parentId].filter(id => id !== systemId);
                if (activeChild[parentId] === systemId) {
                    activeChild[parentId] = null;
                }
            }
            delete parents[systemId];
            // Remove any children references
            if (children[systemId]) {
                for (const childId of children[systemId]) {
                    delete parents[childId];
                    if (!rootSystems.includes(childId) && systems[childId]) {
                        rootSystems.push(childId);
                    }
                }
                delete children[systemId];
            }
            delete activeChild[systemId];
        }

        // --- Hierarchy ---

        function setParent(childId, parentId) {
            // Remove child from root list
            rootSystems = rootSystems.filter(id => id !== childId);
            // Remove from previous parent if any
            const prevParent = parents[childId];
            if (prevParent && children[prevParent]) {
                children[prevParent] = children[prevParent].filter(id => id !== childId);
            }
            // Set new parent
            parents[childId] = parentId;
            if (!children[parentId]) {
                children[parentId] = [];
            }
            if (!children[parentId].includes(childId)) {
                children[parentId].push(childId);
            }
        }

        function hasParent(systemId) {
            return !!parents[systemId];
        }

        // --- Active Child ---

        function setActiveChild(parentId, childId) {
            const prevChild = activeChild[parentId];
            if (prevChild === childId) return;

            // Deactivate previous child
            if (prevChild && systems[prevChild]) {
                activeChild[parentId] = null;
                _notifyDeactivation(prevChild);
            }

            // Activate new child
            if (childId) {
                activeChild[parentId] = childId;
                _notifyActivation(childId);
            }
        }

        function deactivateChild(childId) {
            const parentId = parents[childId];
            if (parentId && activeChild[parentId] === childId) {
                activeChild[parentId] = null;
                _notifyDeactivation(childId);
                // Re-initialize parent so its zone focus is visible
                if (systems[parentId] && systems[parentId].initialize) {
                    systems[parentId].initialize();
                }
            }
        }

        function hasActiveChild(parentId) {
            return !!activeChild[parentId];
        }

        function getActiveChild(parentId) {
            return activeChild[parentId] || null;
        }

        function isActive(systemId) {
            // A system is active if:
            // 1. It has no parent (root) and has DOM-present zones, OR
            // 2. Its parent's activeChild is this system
            const parentId = parents[systemId];
            if (!parentId) {
                return _hasAnyZoneInDOM(systemId);
            }
            return activeChild[parentId] === systemId;
        }

        // --- Event Dispatch ---

        function _ensureListener() {
            if (_listener) return;
            _listener = function(e) { _handleKeydown(e); };
            document.addEventListener('keydown', _listener);
        }

        function _handleKeydown(e) {
            // Build active chain from deepest leaf to root
            const chain = _buildActiveChain();
            if (chain.length === 0) return;

            for (const systemId of chain) {
                const system = systems[systemId];
                if (!system || !system.handle) continue;

                const result = system.handle(e);
                if (result && result.handled) {
                    if (result.preventDefault) e.preventDefault();
                    if (result.stopPropagation) e.stopPropagation();
                    return;
                }
            }
        }

        function _buildActiveChain() {
            // Returns array ordered leaf-first: [deepestChild, ..., root]
            const chain = [];
            for (const rootId of rootSystems) {
                if (!_hasAnyZoneInDOM(rootId)) continue;
                const subchain = [];
                _walkActiveChain(rootId, subchain);
                // subchain is root-first, reverse to get leaf-first
                subchain.reverse();
                chain.push(...subchain);
            }
            return chain;
        }

        function _walkActiveChain(systemId, result) {
            result.push(systemId);
            const childId = activeChild[systemId];
            if (childId && systems[childId]) {
                _walkActiveChain(childId, result);
            }
        }

        // --- DOM Checks ---

        function _hasAnyZoneInDOM(systemId) {
            const system = systems[systemId];
            if (!system || !system.config) return false;
            return system.config.zones.some(z => document.getElementById(z.id));
        }

        function validateActiveChildren() {
            for (const parentId of Object.keys(activeChild)) {
                const childId = activeChild[parentId];
                if (childId && !_hasAnyZoneInDOM(childId)) {
                    activeChild[parentId] = null;
                }
            }
        }

        // --- Activation Callbacks ---

        function _notifyActivation(systemId) {
            const system = systems[systemId];
            if (system && system.onActivate) {
                system.onActivate();
            }
        }

        function _notifyDeactivation(systemId) {
            const system = systems[systemId];
            if (system && system.onDeactivate) {
                system.onDeactivate();
            }
        }

        // --- Public API ---

        return {
            register: register,
            unregister: unregister,
            setParent: setParent,
            hasParent: hasParent,
            setActiveChild: setActiveChild,
            deactivateChild: deactivateChild,
            hasActiveChild: hasActiveChild,
            getActiveChild: getActiveChild,
            isActive: isActive,
            validateActiveChildren: validateActiveChildren,
            // Expose internals for debugging
            _systems: systems,
            _children: children,
            _parents: parents,
            _activeChild: activeChild,
            _rootSystems: rootSystems,
        };
    })();
}
'''.strip()

In [ ]:
# Test coordinator JS generation
result = js_coordinator_setup()
assert "window.kbCoordinator" in result
assert "register" in result
assert "setParent" in result
assert "setActiveChild" in result
assert "deactivateChild" in result
assert "_handleKeydown" in result
assert "_buildActiveChain" in result
assert "validateActiveChildren" in result
assert "_notifyActivation" in result
assert "_notifyDeactivation" in result

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()